# Notebook 02 — Eval MenuOcrParser trên Viet-Menu-gemini-VQA

**Mục tiêu:** đo accuracy của `MenuOcrParser` (Python port từ Kotlin) trên 840 ảnh menu Việt có ground truth.

**Dataset:** [5CD-AI/Viet-Menu-gemini-VQA](https://huggingface.co/datasets/5CD-AI/Viet-Menu-gemini-VQA)

**Pipeline:**
1. Download dataset qua `datasets` library.
2. Với mỗi mẫu: lấy `description` (raw OCR text) + extraction (`Danh sách món`, `Giá`).
3. Chạy `parse(description)` từ `menu_parser.py`.
4. Match fuzzy giữa predicted dishes ↔ ground truth dishes.
5. Compute: precision, recall, F1 cho dish detection + price extraction.
6. Error analysis: tìm 10 case parser sai → đưa vào báo cáo + cải tiến regex.

**Output dùng cho báo cáo đồ án:**
- Bảng metric tổng hợp
- 5-10 case fail tiêu biểu
- Khuyến nghị fix nếu accuracy < 80%

## 1. Setup

In [ ]:
!pip install -q datasets pandas matplotlib

In [ ]:
# Clone repo để dùng menu_parser.py (Python port của Kotlin)
import os, sys, subprocess
if not os.path.exists('/content/hotronguoikhiemthi'):
    subprocess.run(['git', 'clone', 'https://github.com/trandz123/hotronguoikhiemthi.git', '/content/hotronguoikhiemthi'], check=True)
sys.path.insert(0, '/content/hotronguoikhiemthi/ml-training/scripts')

import menu_parser
print('menu_parser loaded')

## 2. Download Viet-Menu-gemini-VQA

In [ ]:
from datasets import load_dataset
ds = load_dataset('5CD-AI/Viet-Menu-gemini-VQA', split='train')
print(f'Dataset size: {len(ds)} samples')
print(f'Columns: {ds.column_names}')
print('\nFirst sample structure:')
for k, v in ds[0].items():
    if k == 'image':
        print(f'  {k}: <PIL Image {v.size}>')
    else:
        s = str(v)
        print(f'  {k}: {s[:300]}{"..." if len(s) > 300 else ""}')

## 3. Parse ground truth ra format chuẩn

Dataset có nhiều dòng cho cùng 1 ảnh (mỗi dòng 1 QA pair). Group lại theo ảnh + extract field `Danh sách món` + `Giá` từ extraction string.

In [ ]:
import re
import json
import pandas as pd

def extract_ground_truth(row):
    """Extract dishes + prices từ 1 row.

    Dataset có column 'description' (raw OCR text) và 'extractions' (JSON-like string).
    Field cụ thể có thể là 'conversations' hoặc 'qa' tùy version. Cần inspect dataset.
    """
    # Tìm extraction qua key 'extractions' nếu có, hoặc parse từ conversations
    extractions = row.get('extractions') or row.get('extraction')
    if isinstance(extractions, str):
        try:
            extractions = json.loads(extractions)
        except Exception:
            extractions = None
    dishes, prices = [], []
    if isinstance(extractions, dict):
        dishes = extractions.get('Danh sách món', []) or extractions.get('Danh sach mon', [])
        prices = extractions.get('Giá', []) or extractions.get('Gia', [])
    return dishes, prices

# Inspect 5 sample đầu để xác nhận key
for i in range(min(5, len(ds))):
    row = ds[i]
    dishes, prices = extract_ground_truth(row)
    print(f'Sample {i}: {len(dishes)} dishes, {len(prices)} prices')
    if dishes:
        print('  e.g.', dishes[0], '|', prices[0] if prices else '?')
    else:
        print('  description preview:', str(row.get('description', ''))[:200])

**Note:** Nếu cell trên báo 0 dishes cho mọi sample → cấu trúc dataset khác với assumption. Lúc đó in `ds[0].keys()` và `ds[0]['conversations'][:500]` để xem format thật rồi sửa `extract_ground_truth`.

## 4. Fuzzy match dishes

In [ ]:
def jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def tokens(s: str) -> set:
    s = menu_parser.normalize_dish_name(s)
    return set(t for t in s.split() if len(t) > 1)

def best_match(pred_name: str, gt_dishes: list, threshold=0.4):
    """Tìm GT dish tốt nhất khớp với pred (Jaccard >= threshold).
    Return (gt_index, score) hoặc (-1, 0)."""
    pt = tokens(pred_name)
    best_i, best_s = -1, 0.0
    for i, gt in enumerate(gt_dishes):
        s = jaccard(pt, tokens(gt))
        if s > best_s:
            best_i, best_s = i, s
    return (best_i, best_s) if best_s >= threshold else (-1, best_s)

# Quick sanity
print(best_match('Pho bo', ['Phở bò tái', 'Bún chả', 'Cơm tấm']))
print(best_match('Trà đá', ['Phở bò tái', 'Trà đá']))
print(best_match('xyz random', ['Phở', 'Bún']))

## 5. Run eval

In [ ]:
from collections import defaultdict
from tqdm.auto import tqdm

results = []
for row in tqdm(ds, desc='Eval'):
    description = row.get('description', '') or ''
    gt_dishes, gt_prices = extract_ground_truth(row)
    gt_prices_int = [menu_parser.parse_price_ground_truth(p) for p in gt_prices]

    pred_items = menu_parser.parse(description)

    # Match từng pred → gt
    matched_gt = set()
    tp_dish, tp_price = 0, 0
    for pred in pred_items:
        gt_idx, _ = best_match(pred.name, gt_dishes)
        if gt_idx >= 0 and gt_idx not in matched_gt:
            matched_gt.add(gt_idx)
            tp_dish += 1
            if pred.price_vnd is not None and gt_idx < len(gt_prices_int):
                gt_price = gt_prices_int[gt_idx]
                if gt_price is not None and abs(pred.price_vnd - gt_price) < 1000:
                    tp_price += 1
    n_pred = len(pred_items)
    n_gt = len(gt_dishes)
    results.append({
        'n_pred': n_pred,
        'n_gt': n_gt,
        'tp_dish': tp_dish,
        'tp_price': tp_price,
        'gt_with_price': sum(1 for p in gt_prices_int if p is not None),
    })

df = pd.DataFrame(results)
tp_d = df['tp_dish'].sum()
tp_p = df['tp_price'].sum()
n_p = df['n_pred'].sum()
n_g = df['n_gt'].sum()
n_gp = df['gt_with_price'].sum()

dish_prec = tp_d / n_p if n_p else 0
dish_rec = tp_d / n_g if n_g else 0
dish_f1 = 2 * dish_prec * dish_rec / (dish_prec + dish_rec) if (dish_prec + dish_rec) else 0
price_acc = tp_p / n_gp if n_gp else 0

print('=== KẾT QUẢ EVAL ===')
print(f'Số ảnh:                       {len(df)}')
print(f'Tổng pred items:              {n_p}')
print(f'Tổng ground truth dishes:     {n_g}')
print(f'Dish detection precision:     {dish_prec:.3f}')
print(f'Dish detection recall:        {dish_rec:.3f}')
print(f'Dish detection F1:            {dish_f1:.3f}')
print(f'Price accuracy (GT có giá):   {price_acc:.3f}')

## 6. Error analysis — 10 case fail tiêu biểu

In [ ]:
# Lọc các sample có recall thấp (parser miss nhiều món)
df['recall'] = df.apply(lambda r: r['tp_dish'] / r['n_gt'] if r['n_gt'] else 1.0, axis=1)
worst = df.nsmallest(10, 'recall')

for idx in worst.index:
    row = ds[int(idx)]
    desc = (row.get('description') or '')[:400]
    gt_d, gt_p = extract_ground_truth(row)
    pred = menu_parser.parse(row.get('description') or '')
    print(f'\n--- Sample idx {idx} (recall {df.loc[idx, "recall"]:.2f}) ---')
    print(f'GT dishes ({len(gt_d)}): {gt_d[:5]}')
    print(f'GT prices ({len(gt_p)}): {gt_p[:5]}')
    print(f'Predicted ({len(pred)}):')
    for p in pred[:5]:
        print(f'  {p.name!r:50s} | {p.price_vnd}')
    print(f'Description preview: {desc[:200]}')

## 7. Lưu kết quả

Export CSV để paste vào báo cáo.

In [ ]:
df.to_csv('/content/menu_eval_results.csv', index=False)

summary = pd.DataFrame([{
    'metric': 'Dish precision', 'value': f'{dish_prec:.3f}',
}, {
    'metric': 'Dish recall', 'value': f'{dish_rec:.3f}',
}, {
    'metric': 'Dish F1', 'value': f'{dish_f1:.3f}',
}, {
    'metric': 'Price accuracy', 'value': f'{price_acc:.3f}',
}, {
    'metric': 'Samples evaluated', 'value': f'{len(df)}',
}])
summary.to_csv('/content/menu_eval_summary.csv', index=False)
print('Saved:')
print('  /content/menu_eval_results.csv (per-sample)')
print('  /content/menu_eval_summary.csv (overall metrics)')

## 8. Khuyến nghị cải tiến parser (sync cả 2 file Kotlin + Python)

Nhìn vào 10 case fail tiêu biểu ở cell 6, common patterns:

| Pattern fail | Fix trong `MenuOcrParser` |
|---|---|
| Giá viết dạng "50." (thiếu chữ số) | Mở rộng regex để chấp nhận trailing dot |
| Tên món dài chia 2 dòng | Merge line liên tiếp nếu line sau không có giá + indent giống line trước |
| Menu có separator giữa tên-giá (vd `:`, `...`, `-`) | Thêm vào `_clean_name` chars |
| Giá dạng range "50-70k" | Lấy giá đầu tiên hoặc giá trung bình |
| Số trong tên món bị nhầm là giá (vd "Cà phê 3in1") | Yêu cầu giá phải ở cuối line hoặc kèm đơn vị |

Sửa file: `app/src/main/kotlin/.../ml/MenuOcrParser.kt` + `ml-training/scripts/menu_parser.py` (2 file phải sync logic).

Sau khi sửa, chạy lại notebook → so sánh F1 trước/sau.